# Benchmark 2.1 — Perbandingan 8 Arsitektur Deep Learning (Sapi Lokal 7-Kelas)

**SapiKenal Research** | Politeknik Negeri Semarang  
**Dataset Target:** [Roboflow Universe `valll-rtclj/sapi-lokal/2`](https://universe.roboflow.com/valll-rtclj/sapi-lokal/dataset/2)  
**7 Kelas:** `aceh`, `bali`, `limusin`, `madura`, `non_sapi`, `pasundan`, `po`

---

## Ringkasan Eksperimen
Notebook ini melakukan evaluasi terstandarisasi terhadap **8 arsitektur deep learning** (Pure CNN dan Hybrid Edge Vision Transformer) pada dataset sistem produksi SapiKenal:
- **Stratified 5-Fold Cross-Validation** dengan augmentasi *on-the-fly*.
- **Mitigasi Severe Class Imbalance:** Penerapan *balanced class weighting* ($w_j = 
rac{N}{K 	imes N_j}$) pada *categorical cross-entropy loss*.
- **2-Phase Transfer Learning:** Feature Extraction (frozen backbone) $\to$ Fine-Tuning (unfreeze top 30%).
- **8 Arsitektur:** MobileNetV3Large, MobileNetV2, EfficientNetB0, DenseNet121, EfficientNetV2S, ConvNeXtTiny, ResNet50, MobileViT-XS.
- **Evaluasi Saintifik:** Macro F1-Score, Per-Class P/R/F1, Uji McNemar $8\times 8$, Visualisasi Grad-CAM, dan Profiling TFLite FP32.


In [ ]:
# ========================================
# 1. Setup Lingkungan & Verifikasi GPU
# ========================================
!pip install -q roboflow transformers scikit-learn seaborn matplotlib pandas

import os, gc, time, math, warnings, pickle, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from collections import Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from scipy import stats as scipy_stats
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical, load_img, img_to_array

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 150
sns.set_style('whitegrid')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Terdeteksi: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("⚠️ GPU tidak terdeteksi — disarankan menggunakan GPU T4 / P100 / A100.")
print(f"TensorFlow Version: {tf.__version__}")


In [ ]:
# ========================================
# 2. Pemuatan, Deduplikasi & Audit Dataset
# ========================================
from roboflow import Roboflow

ROBOFLOW_API_KEY = "oSqbdWgbnJkiVfOWl18o"
WORKSPACE_NAME   = "valll-rtclj"
PROJECT_NAME     = "sapi-lokal"
DATASET_VERSION  = 2

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
dataset = project.version(DATASET_VERSION).download("folder")
DATASET_DIR = dataset.location

IMAGE_SIZE = (224, 224)

raw_paths, raw_labels = [], []
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(DATASET_DIR, split)
    if not os.path.exists(split_dir):
        continue
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for img in sorted(os.listdir(cls_dir)):
            if img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                raw_paths.append(os.path.join(cls_dir, img))
                raw_labels.append(cls)

print(f"Total citra terunduh: {len(raw_paths)}")

DEDUPLICATE_TO_RAW = True
if DEDUPLICATE_TO_RAW and len(raw_paths) > 350:
    base_map = {}
    for p, l in zip(raw_paths, raw_labels):
        fn = os.path.basename(p)
        base_prefix = fn.split(".rf.")[0] if ".rf." in fn else fn
        if base_prefix not in base_map:
            base_map[base_prefix] = (p, l)
    all_paths  = [v[0] for v in base_map.values()]
    all_labels = [v[1] for v in base_map.values()]
    print(f"✅ Deduplikasi aktif: Menggunakan {len(all_paths)} citra asli mentah (un-augmented pool).")
else:
    all_paths  = raw_paths
    all_labels = raw_labels
    print(f"Menggunakan {len(all_paths)} citra.")

class_names  = sorted(set(all_labels))
NUM_CLASSES  = len(class_names)
label_counts = Counter(all_labels)

print(f"\nDistribusi 7 Kelas SapiKenal:")
for c in class_names:
    print(f"  - {c:10s}: {label_counts[c]:3d} citra ({label_counts[c]/len(all_labels)*100:5.2f}%)")

print("\nMemuat seluruh citra ke memori array...")
X_all = np.zeros((len(all_paths), *IMAGE_SIZE, 3), dtype=np.float32)
for i, p in enumerate(all_paths):
    X_all[i] = img_to_array(load_img(p, target_size=IMAGE_SIZE))

label_to_idx = {n: i for i, n in enumerate(class_names)}
y_all        = np.array([label_to_idx[l] for l in all_labels])
y_all_cat    = to_categorical(y_all, NUM_CLASSES)
print(f"Array data siap: X={X_all.shape}, y={y_all.shape}")


In [ ]:
# ========================================
# 3. Parameter Eksperimen & Fungsi Pembantu
# ========================================
BATCH_SIZE         = 16
PHASE1_EPOCHS      = 30
PHASE2_EPOCHS      = 20
PHASE1_LR          = 1e-3
PHASE2_LR          = 1e-5
N_FOLDS            = 5
UNFREEZE_FRACTION  = 0.3
RANDOM_STATE       = 42

OUTPUT_DIR = Path("/content/benchmark2_1_outputs") if os.path.exists("/content") else Path("./benchmark2_1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def build_model(base_fn, preprocess_fn, num_classes, input_shape=(224, 224, 3), name="model"):
    if name == "MobileViT-XS":
        from transformers import TFMobileViTForImageClassification
        m = TFMobileViTForImageClassification.from_pretrained(
            "apple/mobilevit-xs",
            num_labels=num_classes,
            ignore_mismatched_sizes=True,
            id2label={i: c for i, c in enumerate(class_names)},
            label2id={c: i for i, c in enumerate(class_names)},
        )
        m.mobilevit.trainable = False
        m.compile(optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
                  loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                  metrics=["accuracy"])
        return m

    base = base_fn(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x   = GlobalAveragePooling2D(name="gap")(base.output)
    x   = Dense(256, activation="relu", name="dense_head")(x)
    x   = Dropout(0.3, name="dropout_head")(x)
    out = Dense(num_classes, activation="softmax", name="output")(x)
    m   = Model(inputs=base.input, outputs=out, name=name)
    m.compile(optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m

def unfreeze_top(model, fraction=0.3, name="model"):
    if name == "MobileViT-XS":
        model.mobilevit.trainable = True
        model.compile(optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
                      loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                      metrics=["accuracy"])
        return

    base_layers = []
    for layer in model.layers:
        if layer.name == "gap":
            break
        base_layers.append(layer)
    for layer in base_layers:
        layer.trainable = False
    n = max(1, int(len(base_layers) * fraction))
    for layer in base_layers[-n:]:
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True
    for layer in model.layers:
        if layer.name in ("dense_head", "dropout_head", "output"):
            layer.trainable = True
    model.compile(optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
                  loss="categorical_crossentropy", metrics=["accuracy"])

def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=0),
    ]

def eval_model(model, X_val, y_val, preprocess_fn, name="model"):
    X_p = preprocess_fn(X_val.copy())
    if name == "MobileViT-XS":
        logits = model(X_p, training=False).logits
        probs  = tf.nn.softmax(logits, axis=-1).numpy()
    else:
        probs = model.predict(X_p, verbose=0)
    yp = np.argmax(probs, axis=1)
    return {
        "accuracy" : accuracy_score(y_val, yp),
        "precision": precision_score(y_val, yp, average="macro", zero_division=0),
        "recall"   : recall_score(y_val, yp, average="macro", zero_division=0),
        "f1"       : f1_score(y_val, yp, average="macro", zero_division=0),
        "y_pred"   : yp,
        "probs"    : probs,
        "pc_p"     : precision_score(y_val, yp, average=None, zero_division=0),
        "pc_r"     : recall_score(y_val, yp, average=None, zero_division=0),
        "pc_f1"    : f1_score(y_val, yp, average=None, zero_division=0),
    }

print("✅ Setup fungsi pembantu selesai.")


In [ ]:
# ========================================
# 4. Training Loop: 5-Fold CV × 8 Arsitektur × 2 Phase
# ========================================
RESUME = False

MODEL_CONFIGS = {
    "MobileNetV3Large": {
        "base_fn"      : tf.keras.applications.MobileNetV3Large,
        "preprocess_fn": tf.keras.applications.mobilenet_v3.preprocess_input,
    },
    "MobileNetV2": {
        "base_fn"      : tf.keras.applications.MobileNetV2,
        "preprocess_fn": tf.keras.applications.mobilenet_v2.preprocess_input,
    },
    "EfficientNetB0": {
        "base_fn"      : tf.keras.applications.EfficientNetB0,
        "preprocess_fn": tf.keras.applications.efficientnet.preprocess_input,
    },
    "DenseNet121": {
        "base_fn"      : tf.keras.applications.DenseNet121,
        "preprocess_fn": tf.keras.applications.densenet.preprocess_input,
    },
    "EfficientNetV2S": {
        "base_fn"      : tf.keras.applications.EfficientNetV2S,
        "preprocess_fn": tf.keras.applications.efficientnet_v2.preprocess_input,
    },
    "ConvNeXtTiny": {
        "base_fn"      : tf.keras.applications.ConvNeXtTiny,
        "preprocess_fn": tf.keras.applications.convnext.preprocess_input,
    },
    "ResNet50": {
        "base_fn"      : tf.keras.applications.ResNet50,
        "preprocess_fn": tf.keras.applications.resnet50.preprocess_input,
    },
    "MobileViT-XS": {
        "base_fn"      : None,
        "preprocess_fn": lambda x: x / 255.0,
    },
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
ckpt = OUTPUT_DIR / "checkpoint_results_unified.pkl"

if RESUME and ckpt.exists():
    with open(ckpt, "rb") as f:
        all_results = pickle.load(f)
    completed = set(all_results.keys())
    print(f"Resume aktif — sudah selesai: {completed}")
else:
    all_results = {}
    completed   = set()

for model_name, cfg in MODEL_CONFIGS.items():
    if model_name in completed:
        print(f"SKIP {model_name} (sudah selesai)")
        continue

    print(f"\n{'='*70}")
    print(f"  MODEL: {model_name}")
    print(f"{'='*70}")

    mr = {
        "phase1": {"acc": [], "prec": [], "rec": [], "f1": [],
                   "y_trues": [], "y_preds": [], "probs": [],
                   "pc_p": [], "pc_r": [], "pc_f1": []},
        "phase2": {"acc": [], "prec": [], "rec": [], "f1": [],
                   "y_trues": [], "y_preds": [], "probs": [],
                   "pc_p": [], "pc_r": [], "pc_f1": []},
        "times": [], "class_names": class_names,
    }

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all)):
        print(f"  Fold {fold+1}/{N_FOLDS}", end=" ")

        X_tr, X_va = X_all[tr_idx].copy(), X_all[va_idx].copy()
        y_tr_c     = y_all_cat[tr_idx]
        y_va_c     = y_all_cat[va_idx]
        y_tr_l     = y_all[tr_idx]
        y_va_l     = y_all[va_idx]

        classes_unique = np.unique(y_tr_l)
        cw_arr = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_tr_l)
        cw_dict = {i: float(w) for i, w in enumerate(cw_arr)}

        aug = ImageDataGenerator(
            preprocessing_function=cfg["preprocess_fn"],
            rotation_range=25, width_shift_range=0.1, height_shift_range=0.1,
            shear_range=0.1, zoom_range=0.2, horizontal_flip=True,
            brightness_range=[0.8, 1.2], fill_mode="nearest",
        )
        val_gen = ImageDataGenerator(preprocessing_function=cfg["preprocess_fn"])
        steps_tr = math.ceil(len(X_tr) / BATCH_SIZE)
        steps_va = math.ceil(len(X_va) / BATCH_SIZE)

        model = build_model(cfg["base_fn"], cfg["preprocess_fn"],
                            NUM_CLASSES, name=model_name)
        t0 = time.time()

        # Phase 1 — Feature Extraction
        model.fit(
            aug.flow(X_tr, y_tr_c, batch_size=BATCH_SIZE, seed=RANDOM_STATE + fold),
            steps_per_epoch=steps_tr, epochs=PHASE1_EPOCHS,
            validation_data=val_gen.flow(X_va, y_va_c, batch_size=BATCH_SIZE, shuffle=False),
            validation_steps=steps_va, callbacks=get_callbacks(),
            class_weight=cw_dict, verbose=0,
        )
        m1 = eval_model(model, X_va, y_va_l, cfg["preprocess_fn"], name=model_name)
        for k in ["acc", "prec", "rec", "f1"]:
            mr["phase1"][k].append(m1[{"acc":"accuracy","prec":"precision","rec":"recall","f1":"f1"}[k]])
        mr["phase1"]["y_trues"].append(y_va_l)
        mr["phase1"]["y_preds"].append(m1["y_pred"])
        mr["phase1"]["probs"].append(m1["probs"])
        mr["phase1"]["pc_p"].append(m1["pc_p"])
        mr["phase1"]["pc_r"].append(m1["pc_r"])
        mr["phase1"]["pc_f1"].append(m1["pc_f1"])
        print(f"| P1 F1={m1['f1']:.3f} Acc={m1['accuracy']:.3f}", end=" ")

        # Phase 2 — Fine-Tuning
        unfreeze_top(model, UNFREEZE_FRACTION, name=model_name)
        model.fit(
            aug.flow(X_tr, y_tr_c, batch_size=BATCH_SIZE, seed=RANDOM_STATE + fold + 100),
            steps_per_epoch=steps_tr, epochs=PHASE2_EPOCHS,
            validation_data=val_gen.flow(X_va, y_va_c, batch_size=BATCH_SIZE, shuffle=False),
            validation_steps=steps_va, callbacks=get_callbacks(),
            class_weight=cw_dict, verbose=0,
        )
        elapsed = time.time() - t0
        mr["times"].append(elapsed)
        m2 = eval_model(model, X_va, y_va_l, cfg["preprocess_fn"], name=model_name)
        for k in ["acc", "prec", "rec", "f1"]:
            mr["phase2"][k].append(m2[{"acc":"accuracy","prec":"precision","rec":"recall","f1":"f1"}[k]])
        mr["phase2"]["y_trues"].append(y_va_l)
        mr["phase2"]["y_preds"].append(m2["y_pred"])
        mr["phase2"]["probs"].append(m2["probs"])
        mr["phase2"]["pc_p"].append(m2["pc_p"])
        mr["phase2"]["pc_r"].append(m2["pc_r"])
        mr["phase2"]["pc_f1"].append(m2["pc_f1"])
        print(f"| P2 F1={m2['f1']:.3f} Acc={m2['accuracy']:.3f} | {elapsed:.0f}s")

        if m2["f1"] >= max(mr["phase2"]["f1"]):
            if model_name == "MobileViT-XS":
                model.save_pretrained(OUTPUT_DIR / f"best_{model_name}")
            else:
                model.save(OUTPUT_DIR / f"best_{model_name}.keras")

        del model, X_tr, X_va
        tf.keras.backend.clear_session()
        gc.collect()

    all_results[model_name] = mr
    p2_f1 = mr["phase2"]["f1"]
    p2_acc = mr["phase2"]["acc"]
    print(f"  >> {model_name} Final Phase 2: Macro F1={np.mean(p2_f1):.4f} +/- {np.std(p2_f1):.4f} | Acc={np.mean(p2_acc)*100:.2f}%")
    with open(ckpt, "wb") as f:
        pickle.dump(all_results, f)

print("\n✅ SELURUH 8 ARSITEKTUR SELESAI DIEVALUASI!")


In [ ]:
# ========================================
# 5. Profiling Deployment TFLite (FP32)
# ========================================
deployment = {}

for name in MODEL_CONFIGS:
    bp_keras = OUTPUT_DIR / f"best_{name}.keras"
    bp_hf    = OUTPUT_DIR / f"best_{name}"
    
    if not bp_keras.exists() and not bp_hf.exists():
        continue

    print(f"  Profiling TFLite: {name}...", end=" ")
    try:
        if name == "MobileViT-XS":
            from transformers import TFMobileViTForImageClassification
            model = TFMobileViTForImageClassification.from_pretrained(bp_hf)
            converter = tf.lite.TFLiteConverter.from_keras_model(model)
            converter.target_spec.supported_ops = [
                tf.lite.OpsSet.TFLITE_BUILTINS,
                tf.lite.OpsSet.SELECT_TF_OPS
            ]
        else:
            model = tf.keras.models.load_model(bp_keras)
            converter = tf.lite.TFLiteConverter.from_keras_model(model)

        tflite_bytes = converter.convert()
        tflite_path  = OUTPUT_DIR / f"{name}.tflite"
        with open(tflite_path, "wb") as f:
            f.write(tflite_bytes)

        tflite_mb = os.path.getsize(tflite_path) / 1024**2
        keras_mb  = os.path.getsize(bp_keras) / 1024**2 if bp_keras.exists() else 0.0
        params    = model.count_params()

        interp = tf.lite.Interpreter(model_path=str(tflite_path))
        interp.allocate_tensors()
        inp = interp.get_input_details()
        dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
        lats = []
        for _ in range(110):
            t0 = time.time()
            interp.set_tensor(inp[0]["index"], dummy)
            interp.invoke()
            lats.append((time.time() - t0) * 1000)
        avg_lat = np.mean(lats[10:])

        deployment[name] = {
            "params_m"  : params / 1e6,
            "keras_mb"  : keras_mb,
            "tflite_mb" : tflite_mb,
            "latency_ms": avg_lat,
        }
        print(f"{params/1e6:.2f}M | TFLite {tflite_mb:.1f}MB | CPU {avg_lat:.1f}ms")
    except Exception as e:
        print(f"Error {name}: {e}")
        deployment[name] = {"params_m": 0, "keras_mb": 0, "tflite_mb": 0, "latency_ms": 0}

    del model
    tf.keras.backend.clear_session()
    gc.collect()

with open(OUTPUT_DIR / "deployment.pkl", "wb") as f:
    pickle.dump(deployment, f)


In [ ]:
# ========================================
# 6. Pembuatan Tabel Master & Metrik Per-Kelas
# ========================================
MODEL_ORDER = [
    "MobileNetV3Large", "MobileNetV2", "EfficientNetB0", "DenseNet121",
    "EfficientNetV2S", "ConvNeXtTiny", "ResNet50", "MobileViT-XS"
]
MODEL_ORDER = [m for m in MODEL_ORDER if m in all_results]

master_rows = []
for name in MODEL_ORDER:
    p2 = all_results[name]["phase2"]
    dep = deployment.get(name, {})
    master_rows.append({
        "Model"          : name,
        "Macro F1"       : f"{np.mean(p2['f1']):.4f} ± {np.std(p2['f1']):.4f}",
        "Accuracy (%)"   : f"{np.mean(p2['acc'])*100:.2f} ± {np.std(p2['acc'])*100:.2f}",
        "Macro Precision": f"{np.mean(p2['prec']):.4f} ± {np.std(p2['prec']):.4f}",
        "Macro Recall"   : f"{np.mean(p2['rec']):.4f} ± {np.std(p2['rec']):.4f}",
        "Params (M)"     : f"{dep.get('params_m', 0):.2f}",
        "TFLite (MB)"    : f"{dep.get('tflite_mb', 0):.2f}",
        "Latency (ms)"   : f"{dep.get('latency_ms', 0):.2f}",
    })

master_df = pd.DataFrame(master_rows)
print("=" * 95)
print("  TABEL MASTER — BENCHMARK 2.1 (8 ARSITEKTUR, 7 KELAS)")
print("=" * 95)
display(master_df) if 'display' in globals() else print(master_df.to_string(index=False))
master_df.to_csv(OUTPUT_DIR / "benchmark2_1_master_table.csv", index=False)

pc_rows = []
for name in MODEL_ORDER:
    p2 = all_results[name]["phase2"]
    p_mean = np.mean(p2["pc_p"], axis=0)
    r_mean = np.mean(p2["pc_r"], axis=0)
    f_mean = np.mean(p2["pc_f1"], axis=0)
    for i, c in enumerate(class_names):
        pc_rows.append({
            "Model"    : name,
            "Kelas"    : c,
            "Precision": round(float(p_mean[i]), 4),
            "Recall"   : round(float(r_mean[i]), 4),
            "F1_Score" : round(float(f_mean[i]), 4),
        })

pc_df = pd.DataFrame(pc_rows)
pc_df.to_csv(OUTPUT_DIR / "benchmark2_1_per_class_metrics.csv", index=False)
print("✅ Tabel master dan per-kelas disimpan.")


In [ ]:
# ========================================
# 7. Uji Signifikansi Statistik McNemar 8×8
# ========================================
preds_dict = {}
trues_dict = {}
for name in MODEL_ORDER:
    preds_dict[name] = np.concatenate(all_results[name]["phase2"]["y_preds"])
    trues_dict[name] = np.concatenate(all_results[name]["phase2"]["y_trues"])

mcnemar_rows = []
for i in range(len(MODEL_ORDER)):
    for j in range(i + 1, len(MODEL_ORDER)):
        m_a = MODEL_ORDER[i]
        m_b = MODEL_ORDER[j]
        y_true = trues_dict[m_a]
        correct_a = (preds_dict[m_a] == y_true)
        correct_b = (preds_dict[m_b] == y_true)

        b = int(np.sum(correct_a & ~correct_b))
        c = int(np.sum(~correct_a & correct_b))

        if (b + c) == 0:
            p_val = 1.0
        elif (b + c) < 25:
            p_val = float(2 * min(scipy_stats.binom.cdf(min(b, c), b + c, 0.5),
                                  1 - scipy_stats.binom.cdf(min(b, c) - 1, b + c, 0.5)))
        else:
            chi2 = (abs(b - c) - 1)**2 / (b + c)
            p_val = float(1 - scipy_stats.chi2.cdf(chi2, df=1))
        
        p_val = min(1.0, max(0.0, p_val))
        sig = "Signifikan (p < 0.05)" if p_val < 0.05 else "Tidak Signifikan (Setara)"

        mcnemar_rows.append({
            "Model_A"   : m_a,
            "Model_B"   : m_b,
            "b (A+/B-)" : b,
            "c (A-/B+)" : c,
            "p_value"   : round(p_val, 4),
            "Kesimpulan": sig,
        })

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(OUTPUT_DIR / "mcnemar_test_results_8x8.csv", index=False)
display(mcnemar_df) if 'display' in globals() else print(mcnemar_df.to_string(index=False))
print("✅ Uji McNemar 8x8 disimpan ke mcnemar_test_results_8x8.csv")


In [ ]:
# ========================================
# 8. Visualisasi Evaluasi & Deployment Tradeoff
# ========================================
# 1. Confusion Matrix Agregat
n_models = len(MODEL_ORDER)
cols = 4
rows = math.ceil(n_models / cols)
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))
axes = np.array(axes).reshape(-1)

for idx, name in enumerate(MODEL_ORDER):
    ax = axes[idx]
    cm = confusion_matrix(trues_dict[name], preds_dict[name])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    f1_val = np.mean(all_results[name]["phase2"]["f1"])
    ax.set_title(f"{name}\n(Macro F1: {f1_val:.4f})", fontsize=11, fontweight="bold")
    ax.set_ylabel("True", fontsize=9)
    ax.set_xlabel("Pred", fontsize=9)
    ax.tick_params(axis="x", rotation=45)

for idx in range(n_models, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices_7class.png", dpi=200, bbox_inches="tight")
plt.show()

# 2. Scatter Trade-off
fig, ax = plt.subplots(figsize=(9, 6))
xs, ys, sizes, labels = [], [], [], []
for name in MODEL_ORDER:
    f1_mean = np.mean(all_results[name]["phase2"]["f1"])
    dep     = deployment.get(name, {})
    lat     = dep.get("latency_ms", 10.0)
    tfl_mb  = dep.get("tflite_mb", 10.0)
    xs.append(lat)
    ys.append(f1_mean)
    sizes.append(max(50, tfl_mb * 15))
    labels.append(name)

ax.scatter(xs, ys, s=sizes, alpha=0.6, edgecolors="black", c=range(len(labels)), cmap="viridis")
for i, txt in enumerate(labels):
    ax.annotate(f"{txt}\n({deployment.get(txt,{}).get('tflite_mb',0):.1f}MB)",
                (xs[i], ys[i]), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=9, fontweight="bold")

ax.set_title("Pareto Trade-off: Macro F1-Score vs CPU Latency vs Ukuran TFLite (7 Kelas)", fontsize=13, fontweight="bold")
ax.set_xlabel("CPU Latency (ms) [Lebih rendah lebih baik]", fontsize=11)
ax.set_ylabel("Macro F1-Score [Lebih tinggi lebih baik]", fontsize=11)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tradeoff_chart_8models.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# ========================================
# 9. Arsipkan Hasil Lengkap ke ZIP
# ========================================
zip_path = shutil.make_archive("/content/benchmark2_1_all_outputs" if os.path.exists("/content") else "./benchmark2_1_all_outputs", "zip", str(OUTPUT_DIR))
print(f"🎉 SEMUA SELESAI! Seluruh artefak diarsipkan: {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass
